In [1]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)


/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hopper-v4
Loaded 3213 episodes.
2025-07-09 18:04.48 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-09 18:04.48 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-07-09 18:04.48 [info     ] Action size has been automatically determined. action_size=3


In [2]:
import gymnasium as gym
import d3rlpy
import argparse
# parser.add_argument("--dataset", type=str, default="hopper-medium-v0")
# parser.add_argument("--seed", type=int, default=1)
# parser.add_argument("--gpu", type=int)
# parser.add_argument("--compile", action="store_true")
# args = parser.parse_args()

args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = 1
args.compile = False

import gym
env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3600
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

In [3]:
print(env.observation_space)

Box(-inf, inf, (11,), float64)


In [4]:
from d3rlpy.logging import UnifiedFileAdapterFactory
import os
import time

start_time = time.time()


dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=1000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=100000,#100000,
    n_steps_per_epoch=1000,#1000,
    save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=100,
    eval_gaps=1,
    logger_adapter=UnifiedFileAdapterFactory(),
    patience=15,
)


end_time = time.time()
elapsed = end_time - start_time

# Print or log nicely
print(f"\nTotal runtime: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

2025-07-09 18:04.49 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-09 18:04.49 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-07-09 18:04.50 [debug    ] Building models...            
2025-07-09 18:04.50 [debug    ] Models have been built.       
2025-07-09 18:04.50 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450
2025-07-09 18:04.50 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 1000/1000 [00:13<00:00, 74.34it/s, loss=0.659]
/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-07-09 18:05.04 [info     ] New best score                 epoch=1 score=1132.1851175163733
2025-07-09 18:05.04 [info     ] Saving model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450/model_epoch_1.d3' epoch=1
2025-07-09 18:05.04 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450/model_epoch_1.d3
2025-07-09 18:05.04 [info     ] DT_hopper-medium-expert-v2_1_20250709180450: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.0035306706428527832, 'time_algorithm_update': 0.009814193725585937, 'loss': 0.6566235164403915, 'time_step': 0.01340033459663391, 'eval_episode_mean_reward': 1132.1851175163733, 'eval_episode_median_reward': 1132.1851175163733, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': 1132.1851175163733, 'eval_episode_max_reward': 1132.1851175163733, 'eval_episode_count': 1.0} step=1000


Epoch 2/100: 100%|██████████| 1000/1000 [00:12<00:00, 79.88it/s, loss=0.331]


2025-07-09 18:07.04 [info     ] New best score                 epoch=2 score=1438.8428154252217
2025-07-09 18:07.04 [info     ] Saving model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450/model_epoch_2.d3' epoch=2
2025-07-09 18:07.04 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450/model_epoch_2.d3
2025-07-09 18:07.04 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250709180450/model_epoch_1.d3' epoch=1
2025-07-09 18:07.04 [info     ] DT_hopper-medium-expert-v2_1_20250709180450: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.0034651110172271727, 'time_algorithm_update': 0.00895080828666687, 'loss': 0.3307364761531353, 'time_step': 0.012469625473022461, 'eval_episode_mean_reward': 1438.8428154252217, 'eval_episode_median_reward': 1177.1562694746358, 'eval_episode_std_reward': 601.5452516600903, 'eval_episode_min_reward': 905.5199101434963, 'eval_episode_max_reward': 3526.2834805887132, 'eval

...

2025-07-09 19:14.58 [info     ] DT_hopper-medium-expert-v2_1_20250709180450: epoch=26 step=26000 epoch=26 metrics={'time_sample_batch': 0.0033618502616882323, 'time_algorithm_update': 0.008919080018997192, 'loss': 0.19578265604376793, 'time_step': 0.012333349943161011, 'eval_episode_mean_reward': 2277.252881065786, 'eval_episode_median_reward': 2028.4150947506885, 'eval_episode_std_reward': 926.384995767826, 'eval_episode_min_reward': 779.5231370722228, 'eval_episode_max_reward': 3772.2303458975853, 'eval_episode_count': 100.0} step=26000


Epoch 27/100: 100%|██████████| 1000/1000 [00:12<00:00, 81.18it/s, loss=0.194]


2025-07-09 19:18.15 [info     ] DT_hopper-medium-expert-v2_1_20250709180450: epoch=27 step=27000 epoch=27 metrics={'time_sample_batch': 0.0033565893173217775, 'time_algorithm_update': 0.008863382816314698, 'loss': 0.19428623449802399, 'time_step': 0.012272549867630005, 'eval_episode_mean_reward': 2828.7134077155147, 'eval_episode_median_reward': 2774.7440608654324, 'eval_episode_std_reward': 673.7713554931214, 'eval_episode_min_reward': 1492.3402439323606, 'eval_episode_max_reward': 3690.369258027501, 'eval_episode_count': 100.0} step=27000


Epoch 28/100: 100%|██████████| 1000/1000 [00:12<00:00, 81.15it/s, loss=0.194]


2025-07-09 19:21.39 [info     ] DT_hopper-medium-expert-v2_1_20250709180450: epoch=28 step=28000 epoch=28 metrics={'time_sample_batch': 0.0033500425815582275, 'time_algorithm_update': 0.008875718832015992, 'loss': 0.19394282498955726, 'time_step': 0.012277812957763671, 'eval_episode_mean_reward': 2886.604075138095, 'eval_episode_median_reward': 3275.8178920213086, 'eval_episode_std_reward': 795.519275294394, 'eval_episode_min_reward': 770.561357047607, 'eval_episode_max_reward': 3672.945325454876, 'eval_episode_count': 100.0} step=28000
Early stopping at epoch 28 due to no improvement in the last 10 epochs.

Total runtime: 4609.39 seconds (76.82 minutes)


In [ ]:
from d3rlpy.logging import UnifiedFileAdapterFactory
import os
import time

start_time = time.time()


dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=1000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=10000,#100000,
    n_steps_per_epoch=100,#1000,
    save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=1,
    logger_adapter=UnifiedFileAdapterFactory(),

)


end_time = time.time()
elapsed = end_time - start_time

# Print or log nicely
print(f"\nTotal runtime: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

2025-07-08 20:41.34 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-08 20:41.34 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-07-08 20:41.34 [debug    ] Building models...            
2025-07-08 20:41.34 [debug    ] Models have been built.       
2025-07-08 20:41.34 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250708204134
2025-07-08 20:41.34 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 100/100 [00:01<00:00, 55.93it/s, loss=1.64]

2025-07-08 20:41.36 [info     ] New best score                 epoch=1 score=10.7371074068387
2025-07-08 20:41.36 [info     ] Saving model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250708204134/model_epoch_1.d3' epoch=1



/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-07-08 20:41.36 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250708204134/model_epoch_1.d3
2025-07-08 20:41.36 [info     ] DT_hopper-medium-expert-v2_1_20250708204134: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.004193015098571777, 'time_algorithm_update': 0.013518767356872559, 'loss': 1.6111393439769746, 'time_step': 0.017792809009552, 'eval_episode_mean_reward': 10.7371074068387, 'eval_episode_median_reward': 10.7371074068387, 'eval_episode_std_reward': 0.0, 'eval_episode_min_reward': 10.7371074068387, 'eval_episode_max_reward': 10.7371074068387, 'eval_episode_count': 1.0} step=100


Epoch 2/100: 100%|██████████| 100/100 [00:01<00:00, 74.81it/s, loss=1]  


In [4]:
from d3rlpy.logging import UnifiedFileAdapterFactory
import os

dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=1000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=10000,#100000,
    n_steps_per_epoch=100,#1000,
    save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
    logger_adapter=UnifiedFileAdapterFactory(),

)

2025-06-24 11:27.19 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-24 11:27.19 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-24 11:27.20 [debug    ] Building models...            
2025-06-24 11:27.20 [debug    ] Models have been built.       
2025-06-24 11:27.20 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720
2025-06-24 11:27.20 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 100/100 [00:01<00:00, 59.32it/s, loss=1.64]
/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-06-24 11:27.23 [info     ] New best score                 epoch=1 score=10.747784394705855
2025-06-24 11:27.23 [info     ] DT_hopper-medium-expert-v2_1_20250624112720: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.0035964035987854004, 'time_algorithm_update': 0.013100829124450684, 'loss': 1.6111393439769746, 'time_step': 0.016776018142700196, 'eval_episode_mean_reward': 10.747784394705855, 'eval_episode_median_reward': 10.748137937201001, 'eval_episode_std_reward': 0.10659869196591379, 'eval_episode_min_reward': 10.539395464767617, 'eval_episode_max_reward': 10.93364828084358, 'eval_episode_count': 50.0} step=100
2025-06-24 11:27.23 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_1.d3


Epoch 2/100: 100%|██████████| 100/100 [00:01<00:00, 78.43it/s, loss=1]  

2025-06-24 11:27.24 [info     ] DT_hopper-medium-expert-v2_1_20250624112720: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.0035832762718200684, 'time_algorithm_update': 0.009010026454925537, 'loss': 0.9796913182735443, 'time_step': 0.012669634819030762} step=200
2025-06-24 11:27.24 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_2.d3


...

Epoch 59/100: 100%|██████████| 100/100 [00:01<00:00, 70.18it/s, loss=0.241]

2025-06-24 11:37.48 [info     ] DT_hopper-medium-expert-v2_1_20250624112720: epoch=59 step=5900 epoch=59 metrics={'time_sample_batch': 0.004000346660614013, 'time_algorithm_update': 0.010127837657928468, 'loss': 0.24044609054923058, 'time_step': 0.01418745756149292} step=5900


2025-06-24 11:37.48 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_59.d3


Epoch 60/100: 100%|██████████| 100/100 [00:01<00:00, 77.37it/s, loss=0.243]


2025-06-24 11:38.47 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_55.d3' epoch=55
2025-06-24 11:38.47 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_56.d3' epoch=56
2025-06-24 11:38.47 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_57.d3' epoch=57
2025-06-24 11:38.47 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_58.d3' epoch=58
2025-06-24 11:38.47 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624112720/model_epoch_59.d3' epoch=59
2025-06-24 11:38.47 [info     ] DT_hopper-medium-expert-v2_1_20250624112720: epoch=60 step=6000 epoch=60 metrics={'time_sample_batch': 0.0036212825775146485, 'time_algorithm_update': 0.009196438789367677, 'loss': 0.2435320173203945, 'time_step': 0.012869765758514404, 'eval_episode_mean_reward': 1663.970050638132, 'eval

In [4]:
from d3rlpy.logging import UnifiedFileAdapterFactory
import os

dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=1000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=10000,#100000,
    n_steps_per_epoch=100,#1000,
    save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
    logger_adapter=UnifiedFileAdapterFactory(),

)

2025-06-24 11:02.38 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-24 11:02.38 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-24 11:02.38 [debug    ] Building models...            
2025-06-24 11:02.38 [debug    ] Models have been built.       
2025-06-24 11:02.38 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238
2025-06-24 11:02.38 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 100/100 [00:01<00:00, 59.54it/s, loss=1.64]
/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-06-24 11:02.41 [info     ] New best score                 epoch=1 score=10.747784394705855
2025-06-24 11:02.41 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238/model_epoch_1.d3
2025-06-24 11:02.41 [info     ] DT_hopper-medium-expert-v2_1_20250624110238: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.003513603210449219, 'time_algorithm_update': 0.013125181198120117, 'loss': 1.6111393439769746, 'time_step': 0.016717355251312255, 'eval_episode_mean_reward': 10.747784394705855, 'eval_episode_median_reward': 10.748137937201001, 'eval_episode_std_reward': 0.10659869196591379, 'eval_episode_min_reward': 10.539395464767617, 'eval_episode_max_reward': 10.93364828084358, 'eval_episode_count': 50.0} step=100


Epoch 2/100: 100%|██████████| 100/100 [00:01<00:00, 79.62it/s, loss=1]  


2025-06-24 11:02.43 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238/model_epoch_2.d3
2025-06-24 11:02.43 [info     ] DT_hopper-medium-expert-v2_1_20250624110238: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.003498885631561279, 'time_algorithm_update': 0.008916585445404053, 'loss': 0.9796913182735443, 'time_step': 0.012490947246551514} step=200


...

2025-06-24 11:13.09 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238/model_epoch_59.d3
2025-06-24 11:13.09 [info     ] DT_hopper-medium-expert-v2_1_20250624110238: epoch=59 step=5900 epoch=59 metrics={'time_sample_batch': 0.0034754586219787596, 'time_algorithm_update': 0.008567309379577637, 'loss': 0.24044609054923058, 'time_step': 0.012120721340179443} step=5900



Epoch 60/100: 100%|██████████| 100/100 [00:01<00:00, 85.87it/s, loss=0.243]


2025-06-24 11:14.06 [info     ] DT_hopper-medium-expert-v2_1_20250624110238: epoch=60 step=6000 epoch=60 metrics={'time_sample_batch': 0.003490200042724609, 'time_algorithm_update': 0.008013782501220703, 'loss': 0.2435320173203945, 'time_step': 0.011579194068908692, 'eval_episode_mean_reward': 1663.970050638132, 'eval_episode_median_reward': 1068.669479329763, 'eval_episode_std_reward': 1009.8317270371833, 'eval_episode_min_reward': 716.0777973672856, 'eval_episode_max_reward': 3694.1410270112965, 'eval_episode_count': 50.0} step=6000
2025-06-24 11:14.06 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238/model_epoch_55.d3' epoch=55
2025-06-24 11:14.06 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238/model_epoch_56.d3' epoch=56
2025-06-24 11:14.06 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624110238/model_epoch_57.d3' epoch=57
2025-06-24 11:14.06 [info     ] Removing old model 

SystemExit: Early stopping at epoch 60 due to no improvement in the last 10 epochs.

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
from d3rlpy.logging import UnifiedFileAdapterFactory
import os

dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=1000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=10000,#100000,
    n_steps_per_epoch=100,#1000,
    save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
    logger_adapter=UnifiedFileAdapterFactory(),

)

2025-06-24 10:41.43 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-24 10:41.43 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-24 10:41.44 [debug    ] Building models...            
2025-06-24 10:41.44 [debug    ] Models have been built.       
2025-06-24 10:41.44 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144
2025-06-24 10:41.44 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 100/100 [00:01<00:00, 51.36it/s, loss=1.64]
/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-06-24 10:41.47 [info     ] New best score                 epoch=1 score=10.747784394705855
2025-06-24 10:41.47 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_1.d3
2025-06-24 10:41.47 [info     ] DT_hopper-medium-expert-v2_1_20250624104144: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.003551340103149414, 'time_algorithm_update': 0.015753116607666016, 'loss': 1.6111393439769746, 'time_step': 0.019385964870452882, 'eval_episode_mean_reward': 10.747784394705855, 'eval_episode_median_reward': 10.748137937201001, 'eval_episode_std_reward': 0.10659869196591379, 'eval_episode_min_reward': 10.539395464767617, 'eval_episode_max_reward': 10.93364828084358, 'eval_episode_count': 50.0} step=100


Epoch 2/100: 100%|██████████| 100/100 [00:01<00:00, 78.62it/s, loss=1]  

2025-06-24 10:41.48 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_2.d3
2025-06-24 10:41.48 [info     ] DT_hopper-medium-expert-v2_1_20250624104144: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.003535032272338867, 'time_algorithm_update': 0.009034340381622314, 'loss': 0.9796913182735443, 'time_step': 0.012645668983459472} step=200


...

2025-06-24 10:52.13 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_59.d3
2025-06-24 10:52.13 [info     ] DT_hopper-medium-expert-v2_1_20250624104144: epoch=59 step=5900 epoch=59 metrics={'time_sample_batch': 0.00348466157913208, 'time_algorithm_update': 0.008966593742370606, 'loss': 0.24044609054923058, 'time_step': 0.012503573894500733} step=5900


Epoch 60/100: 100%|██████████| 100/100 [00:01<00:00, 79.53it/s, loss=0.243]


2025-06-24 10:53.10 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_37.d3' epoch=37
2025-06-24 10:53.10 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_50.d3' epoch=50
2025-06-24 10:53.10 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_54.d3' epoch=54
2025-06-24 10:53.10 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_56.d3' epoch=56
2025-06-24 10:53.10 [info     ] Removing old model 'd3rlpy_logs/DT_hopper-medium-expert-v2_1_20250624104144/model_epoch_58.d3' epoch=58


SystemExit: Early stopping at epoch 60 due to no improvement in the last 10 epochs.

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from d3rlpy.logging.file_adapter import FileAdapter, FileAdapterFactory
import os

class LightweightFileAdapter(FileAdapter):
    def watch_model(self, epoch: int, step: int) -> None:
        pass  # disable all *_grad.csv logging

class LightweightFileAdapterFactory(FileAdapterFactory):
    def create(self, algo, experiment_name, n_steps_per_epoch):
        logdir = os.path.join(self._root_dir, experiment_name)
        return LightweightFileAdapter(algo, logdir)

dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=1000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=10000,#100000,
    n_steps_per_epoch=100,#1000,
    save_interval=1,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
    logger_adapter=LightweightFileAdapterFactory(),

)

2025-06-23 20:48.34 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-23 20:48.34 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-23 20:48.34 [debug    ] Building models...            
2025-06-23 20:48.34 [debug    ] Models have been built.       
2025-06-23 20:48.34 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250623204834
2025-06-23 20:48.34 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 100/100 [00:01<00:00, 50.23it/s, loss=2.02]
/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-06-23 20:48.40 [info     ] New best score                 epoch=1 score=45.10726467654697
2025-06-23 20:48.40 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250623204834/model_epoch_1.d3
2025-06-23 20:48.40 [info     ] DT_hopper-medium-expert-v2_1_20250623204834: epoch=1 step=100 epoch=1 metrics={'time_sample_batch': 0.003568572998046875, 'time_algorithm_update': 0.0162255859375, 'loss': 1.995084434747696, 'time_step': 0.01985100030899048, 'eval_episode_mean_reward': 45.10726467654697, 'eval_episode_median_reward': 45.5098276611956, 'eval_episode_std_reward': 0.849075426219215, 'eval_episode_min_reward': 43.80363338369836, 'eval_episode_max_reward': 47.32338442955364, 'eval_episode_count': 50.0} step=100


Epoch 2/100: 100%|██████████| 100/100 [00:01<00:00, 77.63it/s, loss=1.2]

2025-06-23 20:48.41 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250623204834/model_epoch_2.d3
2025-06-23 20:48.41 [info     ] DT_hopper-medium-expert-v2_1_20250623204834: epoch=2 step=200 epoch=2 metrics={'time_sample_batch': 0.0036020708084106446, 'time_algorithm_update': 0.009175875186920167, 'loss': 1.1656834548711776, 'time_step': 0.012830309867858887} step=200


...


Epoch 8/100: 100%|██████████| 100/100 [00:01<00:00, 78.01it/s, loss=0.433]

2025-06-23 20:49.23 [info     ] DT_hopper-medium-expert-v2_1_20250623204834: epoch=8 step=800 epoch=8 metrics={'time_sample_batch': 0.0035570311546325685, 'time_algorithm_update': 0.009156978130340577, 'loss': 0.43045501083135607, 'time_step': 0.01276667594909668} step=800



Epoch 9/100: 100%|██████████| 100/100 [00:01<00:00, 78.22it/s, loss=0.414]

2025-06-23 20:49.24 [info     ] DT_hopper-medium-expert-v2_1_20250623204834: epoch=9 step=900 epoch=9 metrics={'time_sample_batch': 0.003543698787689209, 'time_algorithm_update': 0.009134862422943115, 'loss': 0.4129469648003578, 'time_step': 0.012731175422668457} step=900



Epoch 10/100: 100%|██████████| 100/100 [00:01<00:00, 78.02it/s, loss=0.391]


In [3]:
dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=100000,#100000,
    n_steps_per_epoch=1000,#1000,
    save_interval=10,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
)

AttributeError: 'Namespace' object has no attribute 'compile'

In [ ]:
dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=100000,#100000,
    n_steps_per_epoch=1000,#1000,
    save_interval=10,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
)

2025-06-21 16:27.06 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-21 16:27.06 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-21 16:27.07 [debug    ] Building models...            
2025-06-21 16:27.07 [debug    ] Models have been built.       
2025-06-21 16:27.07 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250621162707
2025-06-21 16:27.07 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 1000/1000 [00:19<00:00, 51.34it/s, loss=1.06]
/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-06-21 16:28.10 [info     ] New best score                 epoch=1 score=971.0415869631698
2025-06-21 16:28.10 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250621162707/model_epoch_1.d3
2025-06-21 16:28.10 [info     ] DT_hopper-medium-expert-v2_1_20250621162707: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.005530070543289184, 'time_algorithm_update': 0.01368591809272766, 'loss': 1.0590533695220947, 'time_step': 0.019327185869216917, 'eval_episode_mean_reward': 971.0415869631698, 'eval_episode_median_reward': 956.6371652888727, 'eval_episode_std_reward': 38.25307291179796, 'eval_episode_min_reward': 937.8006344261173, 'eval_episode_max_reward': 1097.4013275522047, 'eval_episode_count': 50.0} step=1000


Epoch 2/100: 100%|██████████| 1000/1000 [00:15<00:00, 63.03it/s, loss=0.524]

2025-06-21 16:28.26 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250621162707/model_epoch_2.d3
2025-06-21 16:28.26 [info     ] DT_hopper-medium-expert-v2_1_20250621162707: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.004774764776229858, 'time_algorithm_update': 0.010867478847503663, 'loss': 0.5234730297923088, 'time_step': 0.015745239973068238} step=2000


...


Epoch 8/100: 100%|██████████| 1000/1000 [00:26<00:00, 38.02it/s, loss=0.266]

2025-06-21 16:32.04 [info     ] DT_hopper-medium-expert-v2_1_20250621162707: epoch=8 step=8000 epoch=8 metrics={'time_sample_batch': 0.008016926527023316, 'time_algorithm_update': 0.017934334993362427, 'loss': 0.26572546021640303, 'time_step': 0.026066546201705933} step=8000



Epoch 9/100: 100%|██████████| 1000/1000 [00:22<00:00, 44.16it/s, loss=0.256]

2025-06-21 16:32.26 [info     ] DT_hopper-medium-expert-v2_1_20250621162707: epoch=9 step=9000 epoch=9 metrics={'time_sample_batch': 0.006860565423965454, 'time_algorithm_update': 0.015481119632720947, 'loss': 0.2560233159661293, 'time_step': 0.022456317901611327} step=9000



Epoch 10/100: 100%|██████████| 1000/1000 [00:15<00:00, 64.15it/s, loss=0.248]


In [4]:
dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=100000,#100000,
    n_steps_per_epoch=1000,#1000,
    save_interval=10,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
)

2025-06-21 13:42.02 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-21 13:42.02 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-21 13:42.03 [debug    ] Building models...            
2025-06-21 13:42.05 [debug    ] Models have been built.       
2025-06-21 13:42.05 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250621134205
2025-06-21 13:42.05 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 1000/1000 [00:28<00:00, 35.35it/s, loss=1.06]

2025-06-21 13:42.34 [info     ] DT_hopper-medium-expert-v2_1_20250621134205: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.007957878828048706, 'time_algorithm_update': 0.019956387996673583, 'loss': 1.0590533695220947, 'time_step': 0.028044097900390624} step=1000



Epoch 2/100: 100%|██████████| 1000/1000 [00:27<00:00, 36.56it/s, loss=0.524]

2025-06-21 13:43.01 [info     ] DT_hopper-medium-expert-v2_1_20250621134205: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.008087256431579589, 'time_algorithm_update': 0.018908551692962648, 'loss': 0.5234730297923088, 'time_step': 0.027119164943695068} step=2000


...

2025-06-21 14:48.25 [info     ] DT_hopper-medium-expert-v2_1_20250621134205: epoch=98 step=98000 epoch=98 metrics={'time_sample_batch': 0.007837825059890747, 'time_algorithm_update': 0.017712752342224122, 'loss': 0.17393746277689934, 'time_step': 0.025672898292541504} step=98000



Epoch 99/100: 100%|██████████| 1000/1000 [00:25<00:00, 38.61it/s, loss=0.173]

2025-06-21 14:48.51 [info     ] DT_hopper-medium-expert-v2_1_20250621134205: epoch=99 step=99000 epoch=99 metrics={'time_sample_batch': 0.007827255725860596, 'time_algorithm_update': 0.01771609854698181, 'loss': 0.17350922764837742, 'time_step': 0.025665363311767577} step=99000



Epoch 100/100: 100%|██████████| 1000/1000 [00:25<00:00, 38.60it/s, loss=0.174]


2025-06-21 14:51.07 [info     ] DT_hopper-medium-expert-v2_1_20250621134205: epoch=100 step=100000 epoch=100 metrics={'time_sample_batch': 0.007831420421600342, 'time_algorithm_update': 0.017716973304748536, 'loss': 0.17388482505083083, 'time_step': 0.02567023801803589, 'eval_episode_mean_reward': 2666.137016434938, 'eval_episode_median_reward': 3311.2217841559877, 'eval_episode_std_reward': 940.4942288395529, 'eval_episode_min_reward': 665.6434922495669, 'eval_episode_max_reward': 3612.021887310709, 'eval_episode_count': 50.0} step=100000
2025-06-21 14:51.07 [info     ] Model parameters are saved to d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250621134205/model_100000.d3


In [ ]:
import gymnasium as gym
gym.make("Walker2d-v4")

In [21]:
!pip list | grep gymnasium
!pip list | grep mujoco

In [12]:
!ls ../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/

d4rl				  hopper-medium-v2.pkl
download_d4rl_datasets.ipynb	  specs_halfcheetah_v2.json
halfcheetah-medium-expert-v2.pkl  specs_hopper_v2.json
halfcheetah-medium-replay-v2.pkl  specs_walker2d_v2.json
halfcheetah-medium-v2.pkl	  walker2d-medium-replay-v2.pkl
hopper-medium-expert-v2.pkl	  walker2d-medium-v2.pkl
hopper-medium-replay-v2.pkl


In [ ]:
halfcheetah-medium-expert-v2
halfcheetah-medium-replay-v2
halfcheetah-medium-v2